In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# =========================
# ===== PARAMETERS ========
# =========================

DATA_DIR = Path(r"C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm")  # ← CHANGE if path changes
CLEAN_SUBDIR = "_cleaned"  # ← CHANGE if folder renamed
CANONICAL_FILENAME = "pak_events_canonical.parquet"  # ← CHANGE if file renamed

# Slice configuration
SLICE_MODE = "first_n_months"  # ← CHANGE: options = "first_n_months", "custom_range"
N_MONTHS = 12  # ← CHANGE if you want 6 months instead of 12

CUSTOM_START = "2017-01-01"  # ← CHANGE (used only if SLICE_MODE="custom_range")
CUSTOM_END   = "2017-12-31"  # ← CHANGE (used only if SLICE_MODE="custom_range")

# Cadence
FREQUENCY = "D"  # ← CHANGE if you ever want weekly ("W") etc.

# =========================

In [2]:
CLEAN_DIR = DATA_DIR / CLEAN_SUBDIR
events_path = CLEAN_DIR / CANONICAL_FILENAME

assert events_path.exists(), f"Missing file: {events_path}"

events = pd.read_parquet(events_path)

# Defensive type enforcement
events["ts"] = pd.to_datetime(events["ts"])
events["date"] = pd.to_datetime(events["date"]).dt.floor("D")
events["user_id"] = (
    events["user_id"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)
events["value"] = pd.to_numeric(events["value"], errors="coerce")

print("Loaded events:", events.shape)

Loaded events: (169787, 6)


In [3]:
full_start = events["date"].min()
full_end   = events["date"].max()

if SLICE_MODE == "first_n_months":
    start_date = full_start
    end_date   = full_start + pd.DateOffset(months=N_MONTHS)
elif SLICE_MODE == "custom_range":
    start_date = pd.Timestamp(CUSTOM_START)
    end_date   = pd.Timestamp(CUSTOM_END)
else:
    raise ValueError("Invalid SLICE_MODE")

end_date = min(end_date, full_end)

print("Full dataset range:", full_start.date(), "->", full_end.date())
print("Chosen slice:", start_date.date(), "->", end_date.date())

Full dataset range: 2016-07-01 -> 2018-08-28
Chosen slice: 2016-07-01 -> 2017-07-01


In [4]:
events_slice = events[
    (events["date"] >= start_date) &
    (events["date"] <= end_date)
].copy()

print("Rows in slice:", len(events_slice))
print("Unique users in slice:", events_slice["user_id"].nunique())

Rows in slice: 94913
Unique users in slice: 36306


In [5]:
daily_index = pd.date_range(start_date, end_date, freq=FREQUENCY)
T = len(daily_index)

print("Cadence:", FREQUENCY)
print("Number of DP releases (T):", T)

Cadence: D
Number of DP releases (T): 366


In [6]:
daily_truth = (
    events_slice
    .groupby("date")
    .agg(
        ORDERS=("order_id", "nunique"),
        REVENUE=("value", "sum"),
        DAU=("user_id", "nunique")
    )
    .reindex(daily_index)
    .fillna(0)
)

print("Zero-order days in slice:", (daily_truth["ORDERS"] == 0).sum())

display(daily_truth.head())
display(daily_truth.describe())

Zero-order days in slice: 0


,ORDERS,REVENUE,DAU
2016-07-01,290,296791.00,103
2016-07-02,114,220574.00,78
2016-07-03,66,80073.00,58
2016-07-04,115,172844.25,77
2016-07-05,65,66167.25,36


,ORDERS,REVENUE,DAU
count,366.000000,3.660000e+02,366.000000
mean,259.325137,8.310808e+05,179.508197
std,466.120933,1.744640e+06,283.198693
min,18.000000,3.012900e+04,17.000000
25%,129.000000,3.501819e+05,102.000000
50%,173.000000,5.237166e+05,126.500000
75%,227.000000,7.285461e+05,159.000000
max,6065.000000,1.827427e+07,3710.000000


Contribution Bounding

In [14]:
# =========================
# ===== DP BOUNDING ======
# =========================

# Max number of orders a user can contribute on any day.
# (Anything beyond this will be clipped.)
MAX_ORDERS_PER_DAY = 1  # ← CHANGE

# Max total spend (revenue) a user can contribute on any day.
# (Anything above this will be clipped.)
MAX_REVENUE_PER_DAY = None  # ← SET below if None, we’ll compute via percentile

# Percentile to determine default B (if MAX_REVENUE_PER_DAY is None)
# e.g., 95 means use 95th percentile of per-user per-day spend
CLIP_PERCENTILE_FOR_B = 90  # ← CHANGE

# Should we bound DAU contributions? (Always 1)
BOUND_DAU = True  # ← CHANGE if you want event-level instead

In [15]:
# --- Compute revenue clipping threshold if not specified ---
if MAX_REVENUE_PER_DAY is None:
    # Compute raw per-user per-day total spend before clipping
    user_day_totals = (
        events_slice
        .groupby(["user_id", "date"])["value"]
        .sum()
        .reset_index(name="daily_spend")
    )
    # Spend threshold B = CLIP_PERCENTILE_FOR_B
    B = user_day_totals["daily_spend"].quantile(CLIP_PERCENTILE_FOR_B / 100)
    MAX_REVENUE_PER_DAY = float(B)
else:
    B = MAX_REVENUE_PER_DAY

print("Using revenue cap B =", B)

Using revenue cap B = 11850.449999999993


In [16]:
# --- Contribution bounding per user per day ---

# 1) Clip orders per day
bounded_orders = (
    events_slice.groupby(["user_id", "date"])["order_id"]
    .nunique()
    .clip(upper=MAX_ORDERS_PER_DAY)
    .reset_index(name="bounded_orders")
)

# 2) Clip revenue per day
bounded_revenue = (
    events_slice.groupby(["user_id", "date"])["value"]
    .sum()
    .clip(upper=MAX_REVENUE_PER_DAY)
    .reset_index(name="bounded_revenue")
)

# 3) Clip DAU presence to 1 per user per day
#    (presence 1 if user has any event(s) that day)
bounded_dau = (
    events_slice.groupby(["user_id", "date"])["user_id"]
    .count()
    .clip(upper=1)
    .reset_index(name="bounded_dau")
)

In [17]:
# --- Reconstruct bounded daily truth series ---

# Merge bounded components
from functools import reduce

dfs = [bounded_orders, bounded_revenue, bounded_dau]
daily_bounded = reduce(
    lambda left, right: pd.merge(left, right, on=["user_id", "date"], how="outer"),
    dfs
).fillna(0)

# Aggregate per day
daily_bounded_truth = (
    daily_bounded
    .groupby("date")
    .agg(
        ORDERS=("bounded_orders", "sum"),
        REVENUE=("bounded_revenue", "sum"),
        DAU=("bounded_dau", "sum"),
    )
    .reindex(daily_index)  # ensure full daily range
    .fillna(0)
)

display(daily_bounded_truth.head())
display(daily_bounded_truth.describe())

,ORDERS,REVENUE,DAU
2016-07-01,103,190990.25,103
2016-07-02,78,122236.80,78
2016-07-03,58,62923.45,58
2016-07-04,77,147658.05,77
2016-07-05,36,58647.70,36


,ORDERS,REVENUE,DAU
count,366.000000,3.660000e+02,366.000000
mean,179.508197,4.878818e+05,179.508197
std,283.198693,7.856294e+05,283.198693
min,17.000000,3.012900e+04,17.000000
25%,102.000000,2.559738e+05,102.000000
50%,126.500000,3.460329e+05,126.500000
75%,159.000000,4.631151e+05,159.000000
max,3710.000000,9.961694e+06,3710.000000


In [18]:
diff_orders = daily_truth["ORDERS"] - daily_bounded_truth["ORDERS"]
print("Days where bounded ORDERS < raw ORDERS:", (diff_orders > 0).sum())
print("Max reduction in a single day:", diff_orders.max())
print("Average reduction:", diff_orders.mean())

Days where bounded ORDERS < raw ORDERS: 360
Max reduction in a single day: 2355
Average reduction: 79.81693989071039


In [19]:
diff_revenue = daily_truth["REVENUE"] - daily_bounded_truth["REVENUE"]
print("Days where bounded REVENUE < raw REVENUE:", (diff_revenue > 0).sum())
print("Max reduction in a single day:", diff_revenue.max())
print("Average reduction:", diff_revenue.mean())

Days where bounded REVENUE < raw REVENUE: 360
Max reduction in a single day: 17947739.0
Average reduction: 343199.019590164


In [20]:
diff_dau = daily_truth["DAU"] - daily_bounded_truth["DAU"]
print("Days where DAU differs:", (diff_dau != 0).sum())
print("Any DAU reduction at all?:", diff_dau.max())

Days where DAU differs: 0
Any DAU reduction at all?: 0


In [21]:
import json
from datetime import datetime
from pathlib import Path

# --- PARAMETERS (same ones you use in this DP notebook) ---
# these should match whatever you set above in DP.ipynb

DATA_DIR = Path(r"C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm")
DP_SUBDIR = "dp_ready"  # folder to store DP postprocessed data
Path(DATA_DIR / DP_SUBDIR).mkdir(parents=True, exist_ok=True)

# These parameters should already exist in your namespace:
# start_date, end_date, FREQUENCY, MAX_ORDERS_PER_DAY,
# MAX_REVENUE_PER_DAY, CLIP_PERCENTILE_FOR_B, T

params = {
    "slice_start": str(start_date.date()),
    "slice_end":   str(end_date.date()),
    "frequency":   FREQUENCY,
    "T":           T,
    "max_orders_per_day": MAX_ORDERS_PER_DAY,
    "clip_pct_for_B":     CLIP_PERCENTILE_FOR_B,
    "revenue_cap_B":      MAX_REVENUE_PER_DAY,
    "generated_on":       datetime.now().strftime("%Y%m%d%H%M%S"),
}

# create a short tag for filename
param_tag_parts = [
    f"{params['slice_start']}",
    f"{params['slice_end']}",
    f"freq={params['frequency']}",
    f"ordk={params['max_orders_per_day']}",
    f"revpct={params['clip_pct_for_B']}",
]
param_tag = "__".join(param_tag_parts)

# safe filename
filename = f"bounded_{param_tag}.parquet"
full_path = DATA_DIR / DP_SUBDIR / filename

print("Saving bounded series to:", full_path)

# save the bounded series
daily_bounded_truth.to_parquet(full_path, index=True)

# also save params metadata as json alongside (optional)
meta_path = full_path.with_suffix(".json")
with open(meta_path, "w") as f:
    json.dump(params, f, indent=2)

print("Also saved params metadata to:", meta_path)

# expose the saved filename
full_path

Saving bounded series to: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\dp_ready\bounded_2016-07-01__2017-07-01__freq=D__ordk=1__revpct=90.parquet
Also saved params metadata to: C:\Users\Siddhartha\Sem8\MajorProject\Experiments\EComm\dp_ready\bounded_2016-07-01__2017-07-01__freq=D__ordk=1__revpct=90.json


WindowsPath('C:/Users/Siddhartha/Sem8/MajorProject/Experiments/EComm/dp_ready/bounded_2016-07-01__2017-07-01__freq=D__ordk=1__revpct=90.parquet')